# Final Project Part II — Real-Time Polymarket Data Analysis Simulation
# Kafka data consumer and MongoDB persistance pipeline

### Claudio Xavier Bayro Jablonski

This notebook implements the Spark Structured Streaming consumer for the final project. It reads simulated Polymarket transaction events from a Kafka topic, parses the JSON records, applies transformations and aggregations, and persists the results into MongoDB using `foreachBatch`.

In [3]:
from pcamarillor.spark_utils import SparkUtils
from pathlib import Path
import shutil
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

kafka_connector = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0"
mongo_connector = "org.mongodb.spark:mongo-spark-connector_2.13:10.4.0"

su = SparkUtils(
    "Final Project - Polymarket Streaming",
    "spark://spark-master:7077",
    spark_packages=f"{kafka_connector},{mongo_connector}"
)

su.spark

## Spark Consumer Notebook

This section creates a Structured Streaming connection to Kafka. The consumer application reads real-time events produced by the polymarket_producer.py script from the polymarket-events topic. Each Kafka message contains a JSON document representing a simulated Polymarket transaction event.

## Read Stream from Kafka

In [14]:
kafka_topic = "polymarket-events"
kafka_bootstrap_servers = "kafka:9093"

raw_kafka_df = (
    su.spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
    .option("subscribe", kafka_topic)
    .option("startingOffsets", "latest")
    .load()
)

raw_kafka_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

Writing category summary batch_id: 6


26/05/10 19:07:27 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing raw events batch_id: 4
Writing trade side summary batch_id: 7
Writing category summary batch_id: 7


26/05/10 19:07:30 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:07:30 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:07:30 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:07:32 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing trade side summary batch_id: 8
Writing category summary batch_id: 8


...

26/05/10 19:08:10 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:08:10 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:08:10 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:08:11 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:08:11 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing trade side summary batch_id: 24
Writing category summary batch_id: 24
Writing raw events batch_id: 13
Writing trade side summary batch_id: 25
Writing category summary batch_id: 25


26/05/10 19:08:15 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:08:15 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:08:15 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:08:16 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:08:16 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing trade side summary batch_id: 26
Writing category summary batch_id: 26
Writing raw events batch_id: 14
Writing category summary batch_id: 27
Writing trade side summary batch_id: 27


26/05/10 19:08:18 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:08:18 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:08:18 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
[Stage 126:>                                                        (0 + 1) / 1]

## Build Transformations

The Kafka message value is converted from binary to string and parsed as JSON. The stream is transformed into structured columns such as market ID, category, trade side, price, quantity, transaction value, and event timestamp.

In [15]:
event_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("market_id", StringType(), True),
    StructField("market_title", StringType(), True),
    StructField("category", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("outcome", StringType(), True),
    StructField("trade_side", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("transaction_value", DoubleType(), True),
    StructField("timestamp", StringType(), True)
])

json_df = raw_kafka_df.selectExpr("CAST(value AS STRING) AS json_value")

parsed_df = (
    json_df
    .withColumn("data", F.from_json(F.col("json_value"), event_schema))
    .select("data.*")
    .withColumn("event_timestamp", F.to_timestamp(F.col("timestamp"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("processing_time", F.current_timestamp())
    .filter(F.col("event_id").isNotNull())
    .filter(F.col("event_timestamp").isNotNull())
)

parsed_df.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- market_id: string (nullable = true)
 |-- market_title: string (nullable = true)
 |-- category: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- outcome: string (nullable = true)
 |-- trade_side: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- transaction_value: double (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- processing_time: timestamp (nullable = false)



26/05/10 19:08:19 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:08:19 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing trade side summary batch_id: 28
Writing category summary batch_id: 28
Writing raw events batch_id: 15
Writing trade side summary batch_id: 29
Writing category summary batch_id: 29


26/05/10 19:08:22 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:08:22 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:08:22 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:08:23 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:08:23 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing trade side summary batch_id: 30
Writing category summary batch_id: 30
Writing raw events batch_id: 16
Writing trade side summary batch_id: 31
Writing category summary batch_id: 31


...

26/05/10 19:08:59 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:08:59 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:08:59 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:09:00 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:09:00 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing trade side summary batch_id: 48
Writing category summary batch_id: 48
Writing raw events batch_id: 25
Writing trade side summary batch_id: 49
Writing category summary batch_id: 49


26/05/10 19:09:02 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:09:02 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:09:02 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:09:03 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:09:03 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing trade side summary batch_id: 50
Writing category summary batch_id: 50
Writing raw events batch_id: 26
Writing category summary batch_id: 51
Writing trade side summary batch_id: 51


26/05/10 19:09:06 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:09:06 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:09:06 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


## Create Aggregated Streaming DataFrames

In [16]:
category_summary_df = (
    parsed_df
    .withWatermark("event_timestamp", "1 minute")
    .groupBy(
        F.window(F.col("event_timestamp"), "1 minute"),
        F.col("category")
    )
    .agg(
        F.count("*").alias("total_trades"),
        F.sum("transaction_value").alias("total_volume"),
        F.avg("price").alias("avg_price"),
        F.sum("quantity").alias("total_quantity")
    )
    .withColumn("window_start", F.col("window.start"))
    .withColumn("window_end", F.col("window.end"))
    .drop("window")
)

trade_side_summary_df = (
    parsed_df
    .withWatermark("event_timestamp", "1 minute")
    .groupBy(
        F.window(F.col("event_timestamp"), "1 minute"),
        F.col("trade_side")
    )
    .agg(
        F.count("*").alias("total_trades"),
        F.sum("transaction_value").alias("total_volume")
    )
    .withColumn("window_start", F.col("window.start"))
    .withColumn("window_end", F.col("window.end"))
    .drop("window")
)

26/05/10 19:09:07 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:09:07 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing trade side summary batch_id: 52
Writing category summary batch_id: 52
Writing raw events batch_id: 27
Writing trade side summary batch_id: 53
Writing category summary batch_id: 53


26/05/10 19:09:08 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:09:08 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:09:08 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:09:09 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:09:09 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing category summary batch_id: 54
Writing trade side summary batch_id: 54
Writing raw events batch_id: 28
Writing trade side summary batch_id: 55
Writing category summary batch_id: 55


26/05/10 19:09:14 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:09:14 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:09:14 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:09:15 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:09:15 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


...

Writing category summary batch_id: 138
Writing trade side summary batch_id: 138
Writing raw events batch_id: 70
Writing trade side summary batch_id: 139
Writing category summary batch_id: 139


26/05/10 19:12:06 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:06 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:06 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:08 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:08 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing trade side summary batch_id: 140
Writing category summary batch_id: 140
Writing raw events batch_id: 71
Writing trade side summary batch_id: 141
Writing category summary batch_id: 141


26/05/10 19:12:12 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:12 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:12 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:14 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:14 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing trade side summary batch_id: 142
Writing category summary batch_id: 142


____________________________________________________________________________________________________________________________________________

## Persist data to MongoDB

The stream is persisted into MongoDB using `foreachBatch`. Each micro-batch is written into a MongoDB collection. The raw event stream is stored in `market_events`, while the aggregated summaries are stored in separate collections for validation and analytics.

In [17]:
mongo_uri = "mongodb://mongodb:27017"
mongo_database = "polymarket"

def write_events_to_mongo(batch_df, batch_id):
    print(f"Writing raw events batch_id: {batch_id}")

    (
        batch_df.write
        .format("mongodb")
        .mode("append")
        .option("spark.mongodb.write.connection.uri", mongo_uri)
        .option("database", mongo_database)
        .option("collection", "market_events")
        .option("writeConcern.w", "majority")
        .save()
    )

def write_category_summary_to_mongo(batch_df, batch_id):
    print(f"Writing category summary batch_id: {batch_id}")

    (
        batch_df.write
        .format("mongodb")
        .mode("append")
        .option("spark.mongodb.write.connection.uri", mongo_uri)
        .option("database", mongo_database)
        .option("collection", "category_summary")
        .option("writeConcern.w", "majority")
        .save()
    )

def write_trade_side_summary_to_mongo(batch_df, batch_id):
    print(f"Writing trade side summary batch_id: {batch_id}")

    (
        batch_df.write
        .format("mongodb")
        .mode("append")
        .option("spark.mongodb.write.connection.uri", mongo_uri)
        .option("database", mongo_database)
        .option("collection", "trade_side_summary")
        .option("writeConcern.w", "majority")
        .save()
    )

Writing raw events batch_id: 72
Writing trade side summary batch_id: 143
Writing category summary batch_id: 143


26/05/10 19:12:18 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:18 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:18 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:20 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:20 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing trade side summary batch_id: 144
Writing category summary batch_id: 144
Writing raw events batch_id: 73
Writing trade side summary batch_id: 145
Writing category summary batch_id: 145


26/05/10 19:12:23 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:23 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:23 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:25 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:25 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing trade side summary batch_id: 146
Writing category summary batch_id: 146
Writing raw events batch_id: 74
Writing trade side summary batch_id: 147
Writing category summary batch_id: 147


...

26/05/10 19:12:44 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:44 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:44 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:46 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:46 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing trade side summary batch_id: 156
Writing category summary batch_id: 156
Writing raw events batch_id: 79
Writing trade side summary batch_id: 157
Writing category summary batch_id: 157


26/05/10 19:12:46 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:46 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:46 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:48 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:48 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing trade side summary batch_id: 158
Writing category summary batch_id: 158
Writing raw events batch_id: 80
Writing category summary batch_id: 159
Writing trade side summary batch_id: 159


26/05/10 19:12:48 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:48 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:12:48 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


## Clean Checkpoints

In [18]:
checkpoint_paths = [
    "/opt/spark/work-dir/checkpoints/polymarket/events",
    "/opt/spark/work-dir/checkpoints/polymarket/category_summary",
    "/opt/spark/work-dir/checkpoints/polymarket/trade_side_summary"
]

for checkpoint_path in checkpoint_paths:
    path = Path(checkpoint_path)
    if path.exists() and path.is_dir():
        shutil.rmtree(path)

print("Checkpoint directories cleaned.")

Checkpoint directories cleaned.


26/05/10 19:12:49 ERROR MicroBatchExecution: Query [id = 846b2b99-0b6b-4422-82d9-f4bf55f875e9, runId = d25cfd6b-6d7f-41ad-8713-c8eb92e1e8a9] terminated with error
java.io.FileNotFoundException: File file:/opt/spark/work-dir/checkpoints/polymarket/events/commits does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.DelegateToFileSystem.getFileStatus(DelegateToFileSystem.java:133)
	at org.apache.hadoop.fs.DelegateToFileSystem.createInternal(DelegateToFileSystem.java:98)
	at org.apache.hadoop.fs.ChecksumFs$ChecksumFSOutputSummer.<init>(ChecksumFs.java:376)
	at org.apache.hadoop.fs.ChecksumFs.createInternal(ChecksumFs.java:423)
	at org.apache.hadoop.fs.AbstractFileSystem.create(AbstractFileSystem.java:638)
	at org.apache.had

## Start Streaming Queries to MongoDB

In [19]:
events_query = (
    parsed_df.writeStream
    .foreachBatch(write_events_to_mongo)
    .outputMode("append")
    .option("checkpointLocation", "/opt/spark/work-dir/checkpoints/polymarket/events")
    .start()
)

category_summary_query = (
    category_summary_df.writeStream
    .foreachBatch(write_category_summary_to_mongo)
    .outputMode("append")
    .option("checkpointLocation", "/opt/spark/work-dir/checkpoints/polymarket/category_summary")
    .start()
)

trade_side_summary_query = (
    trade_side_summary_df.writeStream
    .foreachBatch(write_trade_side_summary_to_mongo)
    .outputMode("append")
    .option("checkpointLocation", "/opt/spark/work-dir/checkpoints/polymarket/trade_side_summary")
    .start()
)

print("Streaming queries started.")
print("Run the producer now in a separate terminal.")

Streaming queries started.
Run the producer now in a separate terminal.


26/05/10 19:13:08 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/10 19:13:08 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/10 19:13:08 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/10 19:13:08 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:08 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:08 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing raw events batch_id: 0
Writing category summary batch_id: 0
Writing trade side summary batch_id: 0
Writing raw events batch_id: 1
Writing trade side summary batch_id: 1
Writing category summary batch_id: 1


26/05/10 19:13:10 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:10 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:10 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:12 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:12 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing category summary batch_id: 2
Writing trade side summary batch_id: 2
Writing raw events batch_id: 2
Writing category summary batch_id: 3
Writing trade side summary batch_id: 3


...

Writing category summary batch_id: 10
Writing trade side summary batch_id: 10
Writing raw events batch_id: 6
Writing trade side summary batch_id: 11
Writing category summary batch_id: 11


26/05/10 19:13:29 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:29 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:29 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:31 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:31 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing trade side summary batch_id: 12
Writing category summary batch_id: 12
Writing raw events batch_id: 7
Writing trade side summary batch_id: 13
Writing category summary batch_id: 13


26/05/10 19:13:34 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:34 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:34 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:36 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:36 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing category summary batch_id: 14
Writing trade side summary batch_id: 14


## Run Stream for Demo Window

In [20]:
events_query.awaitTermination(60)

category_summary_query.stop()
trade_side_summary_query.stop()
events_query.stop()

print("Streaming queries stopped.")

Writing raw events batch_id: 8
Writing trade side summary batch_id: 15
Writing category summary batch_id: 15


26/05/10 19:13:39 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:39 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:39 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:41 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:41 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing category summary batch_id: 16
Writing trade side summary batch_id: 16
Writing raw events batch_id: 9
Writing category summary batch_id: 17
Writing trade side summary batch_id: 17


26/05/10 19:13:42 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:42 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:42 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:44 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:13:44 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing category summary batch_id: 18
Writing trade side summary batch_id: 18
Writing raw events batch_id: 10
Writing category summary batch_id: 19
Writing trade side summary batch_id: 19


...

26/05/10 19:14:03 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:14:03 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:14:03 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:14:05 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:14:05 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing trade side summary batch_id: 28
Writing category summary batch_id: 28
Writing raw events batch_id: 15
Writing trade side summary batch_id: 29
Writing category summary batch_id: 29


26/05/10 19:14:08 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:14:08 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:14:08 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:14:10 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:14:10 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing category summary batch_id: 30
Writing trade side summary batch_id: 30
Streaming queries stopped.


26/05/10 19:14:39 WARN DAGScheduler: Failed to cancel job group 17ab9650-2a94-45b5-9e5c-8a6bdb004cb7. Cannot find active jobs for it.
26/05/10 19:14:39 WARN DAGScheduler: Failed to cancel job group 17ab9650-2a94-45b5-9e5c-8a6bdb004cb7. Cannot find active jobs for it.
26/05/10 19:14:39 WARN DAGScheduler: Failed to cancel job group 17d9f055-01d6-451c-a083-03d3052f2a94. Cannot find active jobs for it.
26/05/10 19:14:39 WARN DAGScheduler: Failed to cancel job group 17d9f055-01d6-451c-a083-03d3052f2a94. Cannot find active jobs for it.
26/05/10 19:14:39 WARN DAGScheduler: Failed to cancel job group 552d423e-670d-456d-958c-43a85d6ae721. Cannot find active jobs for it.
26/05/10 19:14:39 WARN DAGScheduler: Failed to cancel job group 552d423e-670d-456d-958c-43a85d6ae721. Cannot find active jobs for it.


## Validate MongoDB

This section validates that the data was successfully persisted into MongoDB. The validation reads the `market_events`, `category_summary`, and `trade_side_summary` collections and displays the records written by the streaming pipeline.

## Read Raw Events from MongoDB

In [13]:
market_events_df = (
    su.spark.read
    .format("mongodb")
    .option("spark.mongodb.read.connection.uri", mongo_uri)
    .option("database", mongo_database)
    .option("collection", "market_events")
    .load()
)

print("MongoDB market_events collection:")
market_events_df.show(10, truncate=False)
market_events_df.printSchema()

MongoDB market_events collection:
+------------------------+----------+------------------------------------+-------------------+----------+--------------------------------------+-------+-----+-----------------------+--------+-------------------+----------+-----------------+--------+
|_id                     |category  |event_id                            |event_timestamp    |market_id |market_title                          |outcome|price|processing_time        |quantity|timestamp          |trade_side|transaction_value|user_id |
+------------------------+----------+------------------------------------+-------------------+----------+--------------------------------------+-------+-----+-----------------------+--------+-------------------+----------+-----------------+--------+
|6a0008274e703b3a027aca79|World News|f77cc4b0-18da-421c-a8a8-f8cb4422f806|2026-05-10 04:23:01|market-006|Will a peace agreement be reached?    |YES    |0.68 |2026-05-10 04:23:01.701|401     |2026-05-10 04:23:01|BUY  

26/05/10 19:07:15 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:07:15 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:07:15 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:07:18 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing trade side summary batch_id: 2
Writing category summary batch_id: 2


26/05/10 19:07:18 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing raw events batch_id: 2
Writing trade side summary batch_id: 3
Writing category summary batch_id: 3


...

26/05/10 19:07:22 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


Writing raw events batch_id: 3
Writing trade side summary batch_id: 5
Writing category summary batch_id: 5


26/05/10 19:07:25 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:07:25 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
26/05/10 19:07:25 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.
[Stage 27:>                                                         (0 + 1) / 1]

Writing trade side summary batch_id: 6


26/05/10 19:07:27 WARN CaseInsensitiveStringMap: Converting duplicated key writeconcern.w into CaseInsensitiveStringMap.


##  Read Category Summary from MongoDB

In [17]:
validation_df = (
    market_events_df
    .groupBy("category")
    .agg(
        F.count("*").alias("total_events"),
        F.sum("transaction_value").alias("total_volume"),
        F.avg("price").alias("average_price")
    )
    .orderBy(F.col("total_volume").desc())
)

validation_df.show(truncate=False)

+----------+------------+------------------+------------------+
|category  |total_events|total_volume      |average_price     |
+----------+------------+------------------+------------------+
|Technology|6           |1299.27           |0.5483333333333333|
|Crypto    |8           |1185.66           |0.64              |
|World News|4           |1005.1500000000001|0.7424999999999999|
|Sports    |5           |711.75            |0.334             |
|Economy   |3           |633.98            |0.7266666666666667|
|Politics  |6           |494.86            |0.48              |
+----------+------------+------------------+------------------+



## Read Trade Side Summary from MongoDB

In [19]:
trade_side_validation_df = (
    market_events_df
    .groupBy("trade_side")
    .agg(
        F.count("*").alias("total_events"),
        F.sum("transaction_value").alias("total_volume"),
        F.avg("price").alias("average_price")
    )
    .orderBy(F.col("total_volume").desc())
)

trade_side_validation_df.show(truncate=False)

+----------+------------+------------+------------------+
|trade_side|total_events|total_volume|average_price     |
+----------+------------+------------+------------------+
|SELL      |15          |2667.6      |0.5193333333333333|
|BUY       |17          |2663.07     |0.6070588235294118|
+----------+------------+------------+------------------+



## MongoDB Aggregation Validation in Spark

In [20]:
print("Validation aggregation from persisted MongoDB data:")

validation_df = (
    market_events_df
    .groupBy("category")
    .agg(
        F.count("*").alias("total_events"),
        F.sum("transaction_value").alias("total_volume"),
        F.avg("price").alias("average_price")
    )
    .orderBy(F.col("total_volume").desc())
)

validation_df.show(truncate=False)

Validation aggregation from persisted MongoDB data:
+----------+------------+------------------+------------------+
|category  |total_events|total_volume      |average_price     |
+----------+------------+------------------+------------------+
|Technology|6           |1299.27           |0.5483333333333333|
|Crypto    |8           |1185.66           |0.64              |
|World News|4           |1005.1500000000001|0.7424999999999999|
|Sports    |5           |711.75            |0.334             |
|Economy   |3           |633.98            |0.7266666666666667|
|Politics  |6           |494.86            |0.48              |
+----------+------------+------------------+------------------+



## Validation Result

The MongoDB validation confirms that the Spark Structured Streaming consumer successfully read Polymarket events from Kafka, transformed the incoming JSON messages, and persisted both raw and aggregated data into MongoDB. The aggregation query confirms that the persisted data can be used for analytical queries such as total event count and transaction volume by market category.